In [85]:
# ============================================================
# UPLOAD FILES ONE BY ONE (GOOGLE COLAB)
# ============================================================

from google.colab import files
import pandas as pd

def upload_single_file(message):
    print("\n" + "="*70)
    print(message)
    print("="*70)

    uploaded = files.upload()

    if len(uploaded) == 0:
        raise Exception("No file uploaded")

    filename = list(uploaded.keys())[0]

    print(f"Loaded: {filename}")

    return pd.read_csv(filename)

mrna_raw = upload_single_file(
    "Upload mRNA file (context1_GE.csv)"
)

meth_raw = upload_single_file(
    "Upload Methylation file (context2_Meth.csv)"
)

mirna_raw = upload_single_file(
    "Upload miRNA file (context3_miRNA.csv)"
)

protein_raw = upload_single_file(
    "Upload Protein file (context4_Protein.csv)"
)

clinical_raw = upload_single_file(
    "Upload Clinical file (Table1Nature.csv)"
)


Upload mRNA file (context1_GE.csv)


Saving context1_GE.csv to context1_GE (5).csv
Loaded: context1_GE (5).csv

Upload Methylation file (context2_Meth.csv)


Saving context2_Meth.csv to context2_Meth (4).csv
Loaded: context2_Meth (4).csv

Upload miRNA file (context3_miRNA.csv)


Saving context3_miRNA.csv to context3_miRNA (4).csv
Loaded: context3_miRNA (4).csv

Upload Protein file (context4_Protein.csv)


Saving context4_Protein.csv to context4_Protein (4).csv
Loaded: context4_Protein (4).csv

Upload Clinical file (Table1Nature.csv)


Saving Table1Nature.csv to Table1Nature (4).csv
Loaded: Table1Nature (4).csv


In [86]:
def tcga_patient_id(x):

    x = str(x)

    x = x.replace(".", "-")

    parts = x.split("-")

    if len(parts) >= 3:
        return "-".join(parts[:3])

    return x

In [87]:
def process_omics(df, name):

    print(f"\nProcessing {name}")

    sample_ids = df.columns[1:]

    sample_ids = [
        tcga_patient_id(x)
        for x in sample_ids
    ]

    X = df.iloc[:, 1:].T

    X.index = sample_ids

    X.columns = (
        df.iloc[:, 0]
        .astype(str)
    )

    X = X.groupby(
        X.index
    ).mean()

    X = X.apply(
        pd.to_numeric,
        errors="coerce"
    )

    print(
        f"Patients={X.shape[0]}, Features={X.shape[1]}"
    )

    return X

In [88]:
mrna = process_omics(
    mrna_raw,
    "mRNA"
)

meth = process_omics(
    meth_raw,
    "Methylation"
)

mirna = process_omics(
    mirna_raw,
    "miRNA"
)

protein = process_omics(
    protein_raw,
    "Protein"
)


Processing mRNA
Patients=348, Features=645

Processing Methylation
Patients=348, Features=574

Processing miRNA
Patients=348, Features=423

Processing Protein
Patients=348, Features=171


In [89]:
clinical_raw["patient_id"] = (
    clinical_raw["Complete TCGA ID"]
    .astype(str)
    .apply(tcga_patient_id)
)

clinical_raw = (
    clinical_raw
    .drop_duplicates(
        subset="patient_id"
    )
    .set_index("patient_id")
)

In [90]:
common_patients = sorted(
    set(mrna.index)
    &
    set(meth.index)
    &
    set(mirna.index)
    &
    set(protein.index)
)

print(
    "\nCommon Patients:",
    len(common_patients)
)


Common Patients: 348


In [92]:
mrna

Unnamed: 0,1,2,3,4,5,6,7,8,9,10,...,636,637,638,639,640,641,642,643,644,645
TCGA-A1-A0SH,0.29275,0.123833,-1.03250,0.313063,0.230667,-1.94825,0.415063,0.644500,0.475667,-0.150917,...,0.54825,0.28000,-0.204375,1.5825,-0.37050,-0.275250,0.485917,0.683500,-0.798917,0.0284
TCGA-A1-A0SJ,0.30675,1.768333,0.34700,2.653313,1.601500,-2.58325,-0.831313,1.631500,1.277667,-0.316750,...,-0.76075,-0.54875,0.634792,1.7275,2.15400,-0.250917,-0.705083,0.247000,-0.499750,2.6238
TCGA-A1-A0SK,0.43375,-0.108167,-1.04950,-3.083062,-1.466500,-3.52175,-1.194563,-0.335833,1.844917,0.004917,...,0.97775,-2.15500,-3.831125,-1.4315,3.17850,7.485583,-0.432417,3.632143,-1.610250,-0.9017
TCGA-A1-A0SO,-0.72800,-0.238667,0.21550,0.869062,-0.354833,-4.44425,1.454938,-2.328667,-0.818833,3.578250,...,-3.14475,0.57400,-4.086792,-2.1255,1.86650,2.703250,-1.110917,3.642214,-1.048083,-1.9603
TCGA-A2-A04N,0.00300,1.856500,0.18500,-1.377938,0.276500,-0.37075,1.818688,2.091667,-0.264583,0.084583,...,0.61925,-2.28625,-0.867875,1.8745,-0.82325,0.529583,1.386083,-0.435143,0.241583,0.6105
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-E2-A1B4,0.09875,-1.259333,-2.19625,2.023063,-0.306500,0.23725,0.795313,0.210333,-0.648833,-0.691217,...,2.28575,-0.43375,0.587792,1.4765,-0.80875,-1.099417,4.731083,-0.488143,-0.144750,1.7553
TCGA-E2-A1B5,0.90750,2.898167,3.09050,-1.503937,-0.340833,-4.70925,1.827437,-2.833833,-0.110333,0.148750,...,0.45375,-3.00800,-3.255875,1.1990,4.15275,3.375083,-1.130250,4.072286,4.568750,-3.0890
TCGA-E2-A1B6,0.74825,2.825500,2.08050,1.672063,1.792167,1.02875,0.348812,1.494333,1.755417,1.988750,...,-2.71975,-0.25450,-0.272625,2.0290,0.05475,-0.182250,3.525417,2.018857,3.175917,1.4586
TCGA-E2-A1BC,-0.70575,-0.286333,-1.62350,3.170438,2.662667,0.82575,3.009438,0.833167,5.561667,-0.847750,...,2.19425,0.81275,0.384125,0.3265,-0.69775,-1.481550,-0.339583,-1.339714,-1.800917,-0.4545


In [93]:
meth

Unnamed: 0,cg18239753,cg08005849,cg13346411,cg09750385,cg03898365,cg06509239,cg17810944,cg21264055,cg13126790,cg03985136,...,cg19904463,cg05684891,cg01830294,cg19764555,cg20322862,cg18123948,cg06288351,cg22919370,cg21572316,cg08997253
TCGA-A1-A0SH,0.285758,0.375244,0.327511,0.253219,0.562512,0.590432,0.340844,0.390295,0.886248,0.230786,...,0.653851,0.198728,0.529355,0.361744,0.795997,0.425543,0.703681,0.358195,0.240352,0.539000
TCGA-A1-A0SJ,0.717957,0.418318,0.219362,0.501650,0.464676,0.252972,0.496468,0.436418,0.887476,0.155942,...,0.602380,0.224793,0.742674,0.400502,0.842861,0.601613,0.754804,0.477662,0.281128,0.746831
TCGA-A1-A0SK,0.741101,0.470580,0.182638,0.223805,0.434458,0.242995,0.196449,0.221587,0.834029,0.199478,...,0.366149,0.180299,0.219370,0.352139,0.707917,0.246221,0.397911,0.219137,0.151226,0.183131
TCGA-A1-A0SO,0.922448,0.928251,0.192650,0.249733,0.891655,0.433942,0.441323,0.283586,0.902252,0.177182,...,0.241623,0.809085,0.326477,0.611131,0.840886,0.346372,0.358557,0.518641,0.210088,0.854141
TCGA-A2-A04N,0.500559,0.405700,0.201802,0.198354,0.472412,0.575054,0.804098,0.665743,0.803433,0.457795,...,0.708715,0.169732,0.697755,0.369960,0.725672,0.555775,0.751516,0.235774,0.251529,0.452878
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-E2-A1B4,0.300735,0.294412,0.319133,0.215728,0.406697,0.274153,0.410933,0.623388,0.524002,0.208497,...,0.688913,0.154221,0.717895,0.310321,0.446524,0.702899,0.709807,0.314226,0.232786,0.345423
TCGA-E2-A1B5,0.396043,0.667787,0.341930,0.211926,0.683004,0.498248,0.524592,0.437216,0.817643,0.200445,...,0.514910,0.267126,0.686165,0.608769,0.591877,0.655729,0.625344,0.315363,0.224178,0.456075
TCGA-E2-A1B6,0.374133,0.665198,0.332733,0.215585,0.564081,0.343713,0.333734,0.349093,0.756782,0.172317,...,0.244105,0.181360,0.551596,0.656009,0.693085,0.409798,0.630477,0.296042,0.192580,0.283528
TCGA-E2-A1BC,0.467713,0.559657,0.326323,0.302049,0.442220,0.262633,0.279715,0.491056,0.846545,0.194343,...,0.752910,0.181522,0.709711,0.411969,0.815661,0.402273,0.672292,0.249888,0.185415,0.326983


In [94]:
mirna

Unnamed: 0,1,2,3,4,5,6,7,8,9,10,...,414,415,416,417,418,419,420,421,422,423
TCGA-A1-A0SH,8.719308,9.398802,8.718794,10.832687,8.918236,6.509733,6.966789,2.739772,7.998472,5.295114,...,1.060961,0.488405,0.488405,0.815113,1.564083,0.815113,2.512431,3.875953,7.620714,10.661839
TCGA-A1-A0SJ,8.818276,9.506411,8.819176,10.933177,7.974784,5.973964,7.618052,3.506308,8.078149,6.176687,...,3.405339,0.955607,2.161159,2.197363,1.071686,3.817424,3.615384,4.758217,6.934474,11.191954
TCGA-A1-A0SK,8.182346,8.867424,8.181907,10.097749,6.656909,7.177179,7.014288,2.401873,8.110416,5.752160,...,2.756960,1.493481,1.625372,1.894566,0.272977,3.923783,4.749812,4.613244,5.634898,10.845732
TCGA-A1-A0SO,7.829835,8.509595,7.851933,8.929637,7.983121,7.060824,7.589435,2.803334,8.497813,6.425044,...,0.498236,0.000000,2.721818,2.948789,0.000000,2.427723,4.163403,5.237965,6.677974,11.364867
TCGA-A2-A04N,7.769108,8.468957,7.794104,9.263531,6.825188,4.754049,5.944611,0.990270,8.022868,4.453607,...,1.477943,2.133530,1.621505,0.291469,0.000000,0.291469,1.909937,2.282198,6.932423,10.737124
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-E2-A1B4,10.072319,10.770326,10.072397,10.857277,8.294590,5.781727,7.748985,2.861964,10.225253,6.940488,...,0.000000,0.902718,1.271451,1.041291,1.041291,2.027930,3.764823,4.198957,6.879790,10.124735
TCGA-E2-A1B5,9.135431,9.816209,9.144808,9.765352,7.934123,6.198583,6.964220,3.688698,9.841646,7.458824,...,2.302417,0.356619,2.258616,2.384654,1.386155,2.344381,3.599086,4.617761,7.110160,10.440488
TCGA-E2-A1B6,9.181602,9.869041,9.179937,10.179674,8.345719,6.764190,6.840274,4.114982,9.894970,8.014672,...,1.016479,0.777319,2.859846,3.071468,2.715196,2.313638,3.312428,4.510153,7.571457,9.958923
TCGA-E2-A1BC,9.295896,9.981773,9.300228,10.371335,8.179770,6.246653,7.500134,2.705799,9.232643,6.556386,...,0.551046,0.551046,2.030199,1.542301,0.000000,1.307047,2.843249,4.203363,7.144044,11.004647


In [95]:
protein

Unnamed: 0,1,2,3,4,5,6,7,8,9,10,...,162,163,164,165,166,167,168,169,170,171
TCGA-A1-A0SH,-1.074653,-0.881219,-0.264436,-0.181409,-1.088819,1.367920,0.198532,2.185577,0.209965,0.243039,...,-0.832365,0.761762,0.902937,-2.267488,-0.158539,-1.122496,-0.820485,-1.323453,0.520681,-1.121162
TCGA-A1-A0SJ,-1.944972,-0.543676,-1.135546,0.447133,1.673301,0.585416,0.130997,-0.161995,-0.053959,-1.574893,...,-0.806159,-0.973122,0.383730,1.047656,-1.155306,0.026483,-0.534722,1.022510,-0.157809,-0.115210
TCGA-A1-A0SK,0.109603,-0.641481,-0.350712,-0.056292,1.458684,0.154215,-0.861321,-0.441023,1.845624,-0.155911,...,1.021462,2.001528,0.457944,0.783708,1.167366,2.434868,2.615178,0.487202,0.072016,0.348638
TCGA-A1-A0SO,0.429426,0.380032,-0.061796,-0.032621,0.893405,-0.752850,-0.202617,-0.872637,0.471327,-0.394862,...,0.692035,1.897734,-0.801639,0.648207,1.146463,1.505362,2.806390,-0.025869,0.653053,1.489538
TCGA-A2-A04N,-1.176358,-0.187532,-0.162418,1.678216,2.074581,1.288628,-0.658875,1.952627,1.911674,-1.304668,...,-1.129935,0.698026,-1.293649,-2.213262,-1.201836,-1.518996,-1.345560,-1.177624,-0.325878,-1.003472
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-E2-A1B4,-0.503367,-0.418590,0.498724,0.373149,0.029131,0.972507,-0.361646,0.708741,0.398028,0.131877,...,-0.128140,0.488639,-1.203086,-1.560470,0.678006,0.470499,-0.733855,-0.466597,0.259805,-0.670265
TCGA-E2-A1B5,-0.837388,-0.834200,-0.470914,-1.101335,-0.563133,-0.818936,1.620720,1.315055,1.674595,1.645894,...,-1.431105,-0.790743,-0.442995,-0.246197,-0.490941,-0.466164,-0.841231,-1.112257,-1.610184,-0.476742
TCGA-E2-A1B6,0.525911,-0.618713,0.208827,-1.115853,-0.142278,-0.570777,-0.470673,0.870022,0.619032,0.337656,...,-0.419489,0.486716,0.168611,0.532671,0.906473,0.636566,-0.652376,0.390656,0.915701,0.510346
TCGA-E2-A1BC,-0.654675,0.386153,-0.516721,1.004470,0.710420,0.294136,0.202929,0.261811,0.307487,0.675808,...,-0.105485,0.149680,-0.930935,-1.440361,-0.306245,-0.461376,-1.024362,-0.466348,0.129891,-0.309493


In [96]:
from pathlib import Path

output_dir = "Part2_Univariate_Analysis"

Path(output_dir).mkdir(
    exist_ok=True
)

In [97]:
datasets = {
    "mRNA": mrna,
    "Methylation": meth,
    "miRNA": mirna,
    "Protein": protein
}

for name, df in datasets.items():

    print("\n")
    print("="*60)
    print(name)
    print("="*60)

    stats = pd.DataFrame({

        "mean": df.mean(),

        "median": df.median(),

        "std": df.std(),

        "min": df.min(),

        "max": df.max(),

        "variance": df.var()

    })

    stats.to_csv(
        f"{output_dir}/{name}_summary_statistics.csv"
    )

    print(stats.head())



mRNA
                mean  median       std       min       max  variance
Unnamed: 0                                                          
1          -0.301454     0.0  1.645316 -7.681750  2.485500  2.707066
2           0.204349     0.0  2.124696 -3.852333  5.018500  4.514331
3           0.295964     0.0  1.865194 -3.216500  5.464500  3.478948
4           0.051840     0.0  1.511345 -3.083062  4.147562  2.284163
5           0.191438     0.0  1.558686 -2.866833  4.407833  2.429502


Methylation
                mean    median       std       min       max  variance
Unnamed: 0                                                            
cg18239753  0.538285  0.546663  0.186515  0.231021  0.922448  0.034788
cg08005849  0.496537  0.468597  0.143432  0.216800  0.928251  0.020573
cg13346411  0.361718  0.313174  0.157558  0.171604  0.896139  0.024825
cg09750385  0.307452  0.266126  0.125595  0.141106  0.854400  0.015774
cg03898365  0.514463  0.500547  0.107207  0.295479  0.907840  0.011493

In [98]:
global_stats = []

for name, df in datasets.items():

    global_stats.append([

        name,

        df.shape[1],

        df.min().min(),

        df.max().max(),

        df.mean().mean(),

        df.std().mean()

    ])

global_stats = pd.DataFrame(

    global_stats,

    columns=[
        "Modality",
        "Features",
        "Min",
        "Max",
        "Mean",
        "Std"
    ]
)

print(global_stats)

global_stats.to_csv(
    f"{output_dir}/global_modality_statistics.csv",
    index=False
)

      Modality  Features        Min        Max          Mean       Std
0         mRNA       645 -10.168250  12.163125  1.717898e-01  1.841341
1  Methylation       574   0.084507   0.993793  5.343033e-01  0.160974
2        miRNA       423   0.000000  13.321264  3.191861e+00  0.675446
3      Protein       171  -5.365854   5.483411  6.492532e-18  0.994439


In [99]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

for name, df in datasets.items():

    features = np.random.choice(
        df.columns,
        size=min(20, len(df.columns)),
        replace=False
    )

    plt.figure(
        figsize=(16,10)
    )

    for i, feature in enumerate(features):

        plt.subplot(4,5,i+1)

        sns.histplot(
            df[feature],
            kde=True
        )

        plt.title(
            str(feature)[:20]
        )

    plt.tight_layout()

    plt.savefig(
        f"{output_dir}/{name}_feature_distributions.png",
        dpi=300
    )

    plt.close()

In [100]:
for name, df in datasets.items():

    values = df.values.flatten()

    plt.figure(figsize=(8,5))

    sns.histplot(
        values,
        bins=50,
        kde=True
    )

    plt.title(
        f"{name} Distribution"
    )

    plt.savefig(
        f"{output_dir}/{name}_global_distribution.png",
        dpi=300
    )

    plt.close()

In [101]:
variance_results = []

for name, df in datasets.items():

    variances = df.var()

    variance_results.append([

        name,

        variances.mean(),

        variances.median(),

        variances.max()

    ])

    top_var = variances.sort_values(
        ascending=False
    )

    top_var.head(50).to_csv(

        f"{output_dir}/{name}_top50_variance_features.csv"
    )

variance_results = pd.DataFrame(

    variance_results,

    columns=[
        "Modality",
        "MeanVariance",
        "MedianVariance",
        "MaxVariance"
    ]
)

variance_results.to_csv(
    f"{output_dir}/variance_summary.csv",
    index=False
)

In [102]:
skew_results = []

for name, df in datasets.items():

    skew_vals = df.skew()

    skew_results.append([

        name,

        skew_vals.mean(),

        skew_vals.median(),

        skew_vals.abs().mean()

    ])

skew_results = pd.DataFrame(

    skew_results,

    columns=[
        "Modality",
        "MeanSkew",
        "MedianSkew",
        "AbsoluteSkew"
    ]
)

print(skew_results)

skew_results.to_csv(
    f"{output_dir}/skewness_summary.csv",
    index=False
)

      Modality  MeanSkew  MedianSkew  AbsoluteSkew
0         mRNA  0.448281    0.390834      0.746687
1  Methylation  0.309161    0.301251      0.882199
2        miRNA  0.431451    0.293323      0.577364
3      Protein  0.287079    0.278467      0.514238


In [103]:
kurtosis_results = []

for name, df in datasets.items():

    k = df.kurtosis()

    kurtosis_results.append([

        name,

        k.mean(),

        k.median()

    ])

kurtosis_results = pd.DataFrame(

    kurtosis_results,

    columns=[
        "Modality",
        "MeanKurtosis",
        "MedianKurtosis"
    ]
)

kurtosis_results.to_csv(
    f"{output_dir}/kurtosis_summary.csv",
    index=False
)

In [104]:
for name, df in datasets.items():

    sample_features = np.random.choice(
        df.columns,
        size=min(20, len(df.columns)),
        replace=False
    )

    plt.figure(
        figsize=(14,6)
    )

    sns.boxplot(
        data=df[sample_features]
    )

    plt.xticks(
        rotation=90
    )

    plt.tight_layout()

    plt.savefig(
        f"{output_dir}/{name}_boxplots.png",
        dpi=300
    )

    plt.close()

In [105]:
for name, df in datasets.items():

    variance = df.var()

    top100 = variance.sort_values(
        ascending=False
    ).head(100)

    pd.DataFrame({

        "Feature": top100.index,

        "Variance": top100.values

    }).to_csv(

        f"{output_dir}/{name}_top100_features.csv",

        index=False
    )

In [106]:
summary = []

for name, df in datasets.items():

    summary.append([

        name,

        df.shape[0],

        df.shape[1],

        df.min().min(),

        df.max().max(),

        df.mean().mean(),

        df.std().mean()

    ])

summary = pd.DataFrame(

    summary,

    columns=[
        "Modality",
        "Patients",
        "Features",
        "Min",
        "Max",
        "Mean",
        "Std"
    ]
)

summary.to_csv(
    f"{output_dir}/final_univariate_summary.csv",
    index=False
)

print(summary)

      Modality  Patients  Features        Min        Max          Mean  \
0         mRNA       348       645 -10.168250  12.163125  1.717898e-01   
1  Methylation       348       574   0.084507   0.993793  5.343033e-01   
2        miRNA       348       423   0.000000  13.321264  3.191861e+00   
3      Protein       348       171  -5.365854   5.483411  6.492532e-18   

        Std  
0  1.841341  
1  0.160974  
2  0.675446  
3  0.994439  
